# Exploratory Data Analysis — Apartment Prices in Poland

**Input:** `data/processed/master_sales_dataset.csv`  
**Purpose:** Understand data quality, distributions, and structure before Bayesian modeling.

### Outline
1. Data Loading & Overview
2. Missing Values
3. Target Variable: `price`
4. City-level Analysis
5. Temporal Analysis (`snapshot_month`)
6. Continuous Features vs. Price
7. Categorical Features
8. Correlation Matrix
9. Key Findings & Modeling Notes

---

[GenAI Declaration]  
Notebook structure and plotting code were assisted by Claude Sonnet 4.6 on 2026-03-18.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

# ── Paths ──────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path('..').resolve()
DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'master_sales_dataset.csv'
FIG_DIR      = PROJECT_ROOT / 'notebooks' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data path    : {DATA_PATH}')

---
## 1. Data Loading & Overview

In [ ]:
df = pd.read_csv(DATA_PATH)

# snapshot_month as ordered categorical
months_sorted = sorted(df['snapshot_month'].unique())
df['snapshot_month'] = pd.Categorical(df['snapshot_month'], categories=months_sorted, ordered=True)

print(f'Shape        : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Months       : {months_sorted[0]}  →  {months_sorted[-1]}  ({len(months_sorted)} snapshots)')
print(f'Cities       : {sorted(df["city"].unique())}')

In [ ]:
df.dtypes.to_frame('dtype')

In [ ]:
df.describe(include='all').T

---
## 2. Missing Values

In [ ]:
missing = (
    df.isnull().sum()
      .rename('n_missing')
      .to_frame()
      .assign(pct_missing=lambda x: x['n_missing'] / len(df) * 100)
      .sort_values('pct_missing', ascending=False)
)
print(missing[missing['n_missing'] > 0].to_string())

In [ ]:
cols_with_missing = missing[missing['n_missing'] > 0].index.tolist()

if cols_with_missing:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(
        cols_with_missing,
        missing.loc[cols_with_missing, 'pct_missing'],
        color=sns.color_palette('muted')[0]
    )
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Values by Column')
    ax.axvline(5, color='red', linestyle='--', linewidth=0.8, label='5% threshold')
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'missing_values.png')
    plt.show()
else:
    print('No missing values found.')

---
## 3. Target Variable: `price`

In [ ]:
print('price summary:')
print(df['price'].describe().to_string())
print(f'\nSkewness : {df["price"].skew():.3f}')
print(f'Kurtosis : {df["price"].kurtosis():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw price
axes[0].hist(df['price'].dropna(), bins=100, color=sns.color_palette('muted')[0], edgecolor='none')
axes[0].set_title('price  (raw)')
axes[0].set_xlabel('PLN')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

# log(price)
log_price = np.log(df['price'].dropna())
axes[1].hist(log_price, bins=100, color=sns.color_palette('muted')[1], edgecolor='none')
axes[1].set_title('log(price)')
axes[1].set_xlabel('log(PLN)')

for ax in axes:
    ax.set_ylabel('Count')

fig.suptitle('Distribution of Apartment Prices', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'price_distribution.png')
plt.show()

print(f'\nlog(price) skewness : {log_price.skew():.3f}')

In [ ]:
# Outlier check: prices outside [1st pct, 99th pct]
p01, p99 = df['price'].quantile([0.01, 0.99])
outliers = df[(df['price'] < p01) | (df['price'] > p99)]
print(f'1st pct : {p01:,.0f} PLN  |  99th pct : {p99:,.0f} PLN')
print(f'Rows outside [p01, p99] : {len(outliers):,}  ({len(outliers)/len(df)*100:.2f}%)')

---
## 4. City-level Analysis

In [ ]:
city_stats = (
    df.groupby('city')['price']
      .agg(['count', 'median', 'mean', 'std'])
      .rename(columns={'count': 'n', 'median': 'median_price', 'mean': 'mean_price', 'std': 'std_price'})
      .sort_values('median_price', ascending=False)
)
city_stats['mean_price']   = city_stats['mean_price'].map('{:,.0f}'.format)
city_stats['median_price'] = city_stats['median_price'].map('{:,.0f}'.format)
city_stats['std_price']    = city_stats['std_price'].map('{:,.0f}'.format)
print(city_stats.to_string())

In [ ]:
city_order = (
    df.groupby('city')['price'].median()
      .sort_values(ascending=False).index.tolist()
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=df, x='city', y='price', order=city_order,
    showfliers=False, palette='muted', ax=ax
)
ax.set_title('Price Distribution by City (outliers hidden)')
ax.set_xlabel('')
ax.set_ylabel('Price (PLN)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'price_by_city.png')
plt.show()

In [ ]:
# Sample count per city
fig, ax = plt.subplots(figsize=(11, 4))
counts = df['city'].value_counts().reindex(city_order)
ax.bar(counts.index, counts.values, color=sns.color_palette('muted')[2])
ax.set_title('Number of Listings per City')
ax.set_ylabel('Count')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'listings_per_city.png')
plt.show()

---
## 5. Temporal Analysis

In [ ]:
# Listings per month
monthly_counts = df.groupby('snapshot_month', observed=True).size()

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(monthly_counts.index.astype(str), monthly_counts.values, color=sns.color_palette('muted')[3])
ax.set_title('Number of Listings per Month')
ax.set_ylabel('Count')
ax.set_xlabel('Month')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'listings_per_month.png')
plt.show()

In [ ]:
# Median price over time — all cities combined
monthly_price = df.groupby('snapshot_month', observed=True)['price'].median()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(monthly_price.index.astype(str), monthly_price.values, marker='o', linewidth=2)
ax.set_title('National Median Price over Time')
ax.set_ylabel('Median Price (PLN)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.2f}M'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'national_median_price_over_time.png')
plt.show()

In [ ]:
# Median price over time — per city
city_monthly = (
    df.groupby(['city', 'snapshot_month'], observed=True)['price']
      .median()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 6))
for city in city_order:
    sub = city_monthly[city_monthly['city'] == city]
    ax.plot(sub['snapshot_month'].astype(str), sub['price'], marker='o', linewidth=1.5, label=city, markersize=4)

ax.set_title('Median Price over Time by City')
ax.set_ylabel('Median Price (PLN)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'city_median_price_over_time.png')
plt.show()

---
## 6. Continuous Features vs. Price

In [ ]:
CONT_FEATURES = [
    'squareMeters', 'rooms', 'floor', 'floorCount',
    'buildYear', 'centreDistance', 'poiCount',
    'schoolDistance', 'clinicDistance', 'restaurantDistance'
]

# Quick distribution overview
df[CONT_FEATURES].describe().T[['count','mean','std','min','50%','max']]

In [ ]:
# Scatter plots vs. log(price) for the two key model predictors
KEY_FEATURES = ['squareMeters', 'centreDistance', 'poiCount', 'rooms']

df_plot = df[KEY_FEATURES + ['price']].dropna().copy()
df_plot['log_price'] = np.log(df_plot['price'])

# sample 10k for speed
sample = df_plot.sample(min(10_000, len(df_plot)), random_state=42)

fig, axes = plt.subplots(1, len(KEY_FEATURES), figsize=(16, 4))
for ax, feat in zip(axes, KEY_FEATURES):
    ax.scatter(sample[feat], sample['log_price'], alpha=0.15, s=5, color=sns.color_palette('muted')[4])
    ax.set_xlabel(feat)
    ax.set_ylabel('log(price)')
    ax.set_title(feat)

fig.suptitle('Key Features vs. log(price)  (10k sample)', fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR / 'key_features_vs_logprice.png')
plt.show()

In [ ]:
# price per sqm — derived feature useful for model checking
df['price_per_sqm'] = df['price'] / df['squareMeters']

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['price_per_sqm'].dropna(), bins=100, color=sns.color_palette('muted')[5], edgecolor='none')
axes[0].set_title('price_per_sqm (raw)')
axes[0].set_xlabel('PLN / m²')

axes[1].hist(np.log(df['price_per_sqm'].dropna()), bins=100, color=sns.color_palette('muted')[0], edgecolor='none')
axes[1].set_title('log(price_per_sqm)')
axes[1].set_xlabel('log(PLN / m²)')

for ax in axes:
    ax.set_ylabel('Count')

plt.tight_layout()
plt.savefig(FIG_DIR / 'price_per_sqm.png')
plt.show()

print('price_per_sqm summary:')
print(df['price_per_sqm'].describe().to_string())

---
## 7. Categorical Features

In [ ]:
CAT_FEATURES = ['type', 'ownership', 'buildingMaterial', 'condition',
                'hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']

for col in CAT_FEATURES:
    print(f'{col:25s}: {df[col].value_counts(dropna=False).to_dict()}')

In [ ]:
# Median price by apartment type and ownership
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col in zip(axes, ['type', 'ownership']):
    order = df.groupby(col)['price'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='price', order=order, showfliers=False, palette='muted', ax=ax)
    ax.set_title(f'Price by {col}')
    ax.set_xlabel('')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig(FIG_DIR / 'price_by_type_ownership.png')
plt.show()

In [ ]:
# Binary amenity features — median price uplift
binary_cols = ['hasParkingSpace', 'hasBalcony', 'hasElevator', 'hasSecurity', 'hasStorageRoom']

rows = []
for col in binary_cols:
    med_yes = df[df[col] == 'yes']['price'].median()
    med_no  = df[df[col] == 'no']['price'].median()
    rows.append({'feature': col, 'median_yes': med_yes, 'median_no': med_no,
                 'uplift_pct': (med_yes - med_no) / med_no * 100})

pd.DataFrame(rows).set_index('feature').round(1)

---
## 8. Correlation Matrix (numeric features)

In [ ]:
NUM_COLS = ['price', 'squareMeters', 'rooms', 'floor', 'floorCount',
            'buildYear', 'centreDistance', 'poiCount',
            'schoolDistance', 'clinicDistance', 'restaurantDistance']

corr = df[NUM_COLS].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    linewidths=0.4, ax=ax
)
ax.set_title('Pearson Correlation Matrix')
plt.tight_layout()
plt.savefig(FIG_DIR / 'correlation_matrix.png')
plt.show()

# Top correlations with price
print('\nCorrelations with price (sorted):')
print(corr['price'].drop('price').sort_values(ascending=False).to_string())

---
## 9. Key Findings & Modeling Notes

In [ ]:
# Summary statistics to inform prior elicitation
log_price = np.log(df['price'].dropna())
log_sqm   = np.log(df['squareMeters'].dropna())

print('=== Prior Elicitation Reference ===')
print(f'log(price)   mean={log_price.mean():.3f}  std={log_price.std():.3f}')
print(f'log(sqm)     mean={log_sqm.mean():.3f}    std={log_sqm.std():.3f}')
print(f'centreDistance  mean={df["centreDistance"].mean():.2f}  std={df["centreDistance"].std():.2f}')
print(f'Number of cities : {df["city"].nunique()}')
print(f'Number of months : {df["snapshot_month"].nunique()}')
print()
print('Per-city mean log(price):')
print(df.groupby('city', observed=True).apply(lambda g: np.log(g['price']).mean()).sort_values(ascending=False).round(3).to_string())

### Summary

| Finding | Implication for Modeling |
|---|---|
| `price` is right-skewed; `log(price)` is approximately Normal | Use **log(price)** as the response variable |
| Strong city-level heterogeneity in median price | Justifies **hierarchical intercepts** per city |
| `squareMeters` has the strongest positive correlation with price | Include as a key covariate; consider log-transform |
| `centreDistance` shows a negative correlation with price | Central predictor for the TVP slope $\beta_{c,t}$ |
| Visible (though small) time trends differ across cities | Justifies **time-varying parameters** (TVP model) |
| `buildYear`, `floorCount`, `poiCount` have moderate correlations | Include as fixed effects in Stage 1 & 2 models |
| Some columns have >30% missing (`condition`, `buildingMaterial`) | Exclude or impute carefully before modeling |